# Dimensionality Reduction — Implementations

LDA and t-SNE once more. LDA's discriminant directions carry two ambiguities — sign and scale — so every lane canonicalises them (unit norm, largest-magnitude entry positive) before exporting, and eigenvalues cross library boundaries as Rayleigh quotients rather than raw solver output. t-SNE gets a tensor mirror driven by the exact same NumPy draws; a fixed 120-iteration budget at a gentle learning rate keeps two float64 implementations on the same trajectory.

## 12_lda

Project where the classes separate, not where the variance is.

### torch

The notebook's exact route on tensors: accumulate S_W and S_B per class, ridge S_W, `torch.linalg.solve`, then a nonsymmetric `torch.linalg.eig`. **What torch adds:** honesty about ambiguity — eig hands back directions with arbitrary sign and scale, so the fixture canonicalises every column before anything is compared.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Accumulate S_W and S_B class by class, in the order np.unique gives you.
# 2. Keep the ridge eps = 1e-6*trace(S_W)/d; dropping it moves every export.
# 3. torch.linalg.eig returns complex pairs; .real mirrors the notebook's np.real step.
# 4. Sort in numpy after .numpy(): argsort descending, then slice the first k columns.
# 5. Directions carry arbitrary sign and scale — canonicalise before comparing anything.


class LDAScratch:
    """Fisher's LDA with the scatter accumulation and the generalized
    eigenproblem on tensors — the notebook's exact route: ridge S_W, then
    solve, then a nonsymmetric eig. Attributes come back as NumPy arrays so
    the fixture code is identical in every lane."""

    def __init__(self, n_components=None):
        self.n_components = n_components

    def fit(self, X, y):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        y = np.asarray(y)
        n, d = Xt.shape
        self.classes_ = np.unique(y)
        C = len(self.classes_)
        if self.n_components is None:
            self.n_components = min(d, C - 1)

        global_mean = Xt.mean(dim=0)
        means = torch.stack([Xt[torch.as_tensor(y == c)].mean(dim=0)
                             for c in self.classes_])

        S_W = torch.zeros((d, d), dtype=torch.float64)
        for c in self.classes_:
            Xc = Xt[torch.as_tensor(y == c)]
            Xc = Xc - Xc.mean(dim=0)
            S_W = S_W + Xc.T @ Xc

        S_B = torch.zeros((d, d), dtype=torch.float64)
        for i, c in enumerate(self.classes_):
            n_c = int(np.sum(y == c))
            diff = (means[i] - global_mean).reshape(-1, 1)
            S_B = S_B + n_c * (diff @ diff.T)

        # Same ridge as the notebook: keeps S_W invertible at a ~1e-6 cost.
        eps = 1e-6 * float(torch.trace(S_W)) / d
        A = torch.linalg.solve(S_W + eps * torch.eye(d, dtype=torch.float64), S_B)
        evals, evecs = torch.linalg.eig(A)  # complex, like np.linalg.eig
        evals = evals.real.numpy()
        evecs = evecs.real.numpy()

        idx = np.argsort(evals)[::-1]
        k = self.n_components
        self.eigenvalues_ = evals[idx][:k]
        self.scalings_ = evecs[:, idx][:, :k]
        self.means_ = means.numpy()
        self.global_mean_ = global_mean.numpy()
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.global_mean_) @ self.scalings_

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)


In [ ]:
# exports: W_lda, evals_lda, M_lda, labels_lda
_rng_lda_eq = np.random.default_rng(1205)
_centers_eq = np.array([[0.0, 0.0, 0.0, 0.0, 0.0],
                        [3.0, 1.0, 0.0, -1.0, 0.5],
                        [-1.0, 2.5, 1.5, 0.5, -0.5]])
X_lda_eq = np.repeat(_centers_eq, 20, axis=0) + _rng_lda_eq.normal(size=(60, 5))
y_lda_eq = np.repeat(np.array([0, 1, 2]), 20)

_lda_eq = LDAScratch(n_components=2).fit(X_lda_eq, y_lda_eq)
W_lda = _lda_eq.scalings_ / np.linalg.norm(_lda_eq.scalings_, axis=0)
W_lda = W_lda * np.sign(W_lda[np.argmax(np.abs(W_lda), axis=0), np.arange(2)])
evals_lda = _lda_eq.eigenvalues_
M_lda = (_lda_eq.means_ - _lda_eq.global_mean_) @ W_lda
_Z_lda_eq = (X_lda_eq - _lda_eq.global_mean_) @ W_lda
labels_lda = np.argmin(((_Z_lda_eq[:, None, :] - M_lda[None, :, :]) ** 2).sum(axis=2), axis=1)
print("eigenvalues:", np.round(evals_lda, 6))
print("canonical directions:")
print(np.round(W_lda, 4))


In [ ]:
# Rebuild the scatter matrices from the fixture and hold the exported
# directions to the eigen equation they claim to solve.
_d_chk = X_lda_eq.shape[1]
_SW_chk = np.zeros((_d_chk, _d_chk))
_SB_chk = np.zeros((_d_chk, _d_chk))
for _c in (0, 1, 2):
    _Xc = X_lda_eq[y_lda_eq == _c] - X_lda_eq[y_lda_eq == _c].mean(axis=0)
    _SW_chk += _Xc.T @ _Xc
    _df = (X_lda_eq[y_lda_eq == _c].mean(axis=0) - X_lda_eq.mean(axis=0)).reshape(-1, 1)
    _SB_chk += (y_lda_eq == _c).sum() * (_df @ _df.T)
_eps_chk = 1e-6 * np.trace(_SW_chk) / _d_chk
_A_chk = np.linalg.solve(_SW_chk + _eps_chk * np.eye(_d_chk), _SB_chk)
_res_chk = max(np.linalg.norm(_A_chk @ W_lda[:, _j] - evals_lda[_j] * W_lda[:, _j])
               for _j in range(2))
assert _res_chk < 1e-8, "each canonical direction is an eigenvector of (S_W+eps I)^-1 S_B"
assert evals_lda[0] >= evals_lda[1] > 0, "Fisher ratios are positive and sorted descending"

# Optimality: no random direction beats the top eigenvalue's Fisher ratio.
_R_chk = _SW_chk + _eps_chk * np.eye(_d_chk)
_best_rand = max((_w @ _SB_chk @ _w) / (_w @ _R_chk @ _w)
                 for _w in np.random.default_rng(5).normal(size=(50, _d_chk)))
assert _best_rand <= evals_lda[0] + 1e-9, "the top direction maximises the Fisher ratio"

_acc_chk = (labels_lda == y_lda_eq).mean()
assert _acc_chk > 0.9, "nearest class mean in LDA space separates the classes"


### library

sklearn's `LinearDiscriminantAnalysis(solver='eigen')` solves `eigh(S_b, S_w)` on covariances — both scatters divided by n, so the eigenpairs match the scratch problem exactly, minus its 1e-6 ridge (a ~1e-7 tilt in the directions, ~1e-5 in raw eigenvalues). **What the library adds:** a symmetric-definite solver, plus the Rayleigh-quotient translation that recovers notebook-convention eigenvalues to ~1e-12 because quotients are stationary at eigenvectors.

In [ ]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# hints:
# 1. solver='eigen' solves eigh(S_b, S_w) on covariances — scatters over n, same eigenpairs.
# 2. scalings_ columns are S_w-normalised (v'S_w v = 1), not unit norm; canonicalise them.
# 3. The raw generalized eigenvalues are not exposed; a Rayleigh quotient recovers them.
# 4. Ridge S_W inside that quotient — stationarity then makes it exact to ~1e-12.


class LDAScratch:
    """sklearn's eigen-solver LDA behind the notebook's interface.

    `solver='eigen'` solves eigh(S_b, S_w) with both scatters divided by n,
    which leaves the eigenvectors of the scratch problem untouched. The raw
    eigenvalues are not exposed, so fit recovers notebook-convention values
    as Fisher ratios of sklearn's own directions under the scratch's ridged
    within-scatter — a translation, not a re-fit."""

    def __init__(self, n_components=None):
        self.n_components = n_components

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        n, d = X.shape
        self.classes_ = np.unique(y)
        C = len(self.classes_)
        if self.n_components is None:
            self.n_components = min(d, C - 1)
        k = self.n_components

        self._model = LinearDiscriminantAnalysis(solver='eigen',
                                                 n_components=k).fit(X, y)
        self.global_mean_ = X.mean(axis=0)
        self.means_ = self._model.means_
        self.scalings_ = self._model.scalings_[:, :k]

        S_W = np.zeros((d, d))
        S_B = np.zeros((d, d))
        for i, c in enumerate(self.classes_):
            Xc = X[y == c] - X[y == c].mean(axis=0)
            S_W += Xc.T @ Xc
            diff = (self.means_[i] - self.global_mean_).reshape(-1, 1)
            S_B += np.sum(y == c) * (diff @ diff.T)
        R = S_W + (1e-6 * np.trace(S_W) / d) * np.eye(d)
        self.eigenvalues_ = np.array([(w @ S_B @ w) / (w @ R @ w)
                                      for w in self.scalings_.T])
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.global_mean_) @ self.scalings_

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)


In [ ]:
# exports: W_lda, evals_lda, M_lda, labels_lda
_rng_lda_eq = np.random.default_rng(1205)
_centers_eq = np.array([[0.0, 0.0, 0.0, 0.0, 0.0],
                        [3.0, 1.0, 0.0, -1.0, 0.5],
                        [-1.0, 2.5, 1.5, 0.5, -0.5]])
X_lda_eq = np.repeat(_centers_eq, 20, axis=0) + _rng_lda_eq.normal(size=(60, 5))
y_lda_eq = np.repeat(np.array([0, 1, 2]), 20)

_lda_eq = LDAScratch(n_components=2).fit(X_lda_eq, y_lda_eq)
W_lda = _lda_eq.scalings_ / np.linalg.norm(_lda_eq.scalings_, axis=0)
W_lda = W_lda * np.sign(W_lda[np.argmax(np.abs(W_lda), axis=0), np.arange(2)])
evals_lda = _lda_eq.eigenvalues_
M_lda = (_lda_eq.means_ - _lda_eq.global_mean_) @ W_lda
_Z_lda_eq = (X_lda_eq - _lda_eq.global_mean_) @ W_lda
labels_lda = np.argmin(((_Z_lda_eq[:, None, :] - M_lda[None, :, :]) ** 2).sum(axis=2), axis=1)
print("eigenvalues (Rayleigh):", np.round(evals_lda, 6))


In [ ]:
# Generalized eigenvectors from eigh(S_b, S_w) are S_W-orthogonal — that is
# the property the symmetric-definite solver guarantees and eig does not.
_d_chk = X_lda_eq.shape[1]
_SW_chk = np.zeros((_d_chk, _d_chk))
for _c in (0, 1, 2):
    _Xc = X_lda_eq[y_lda_eq == _c] - X_lda_eq[y_lda_eq == _c].mean(axis=0)
    _SW_chk += _Xc.T @ _Xc
_v0, _v1 = _lda_eq.scalings_[:, 0], _lda_eq.scalings_[:, 1]
assert abs(_v0 @ _SW_chk @ _v1) < 1e-8 * abs(_v0 @ _SW_chk @ _v0), \
    "eigen-solver directions are S_W-orthogonal"

_k_max = len(_lda_eq.classes_) - 1
assert _lda_eq.scalings_.shape == (5, _k_max), "at most C-1 = 2 discriminant directions"
assert evals_lda[0] >= evals_lda[1] > 0, "Fisher ratios are positive and sorted descending"

_acc_chk = (labels_lda == y_lda_eq).mean()
assert _acc_chk > 0.9, "nearest class mean in LDA space separates the classes"


## 12_tsne

Match neighbour distributions, not distances.

No library lane on purpose: sklearn's `TSNE` runs Barnes–Hut approximation with PCA initialisation and its own exaggeration and learning-rate schedule — a different algorithm whose embedding can only be compared by eye, never by max |delta|, so an honest absence beats a lane that fails its own comparison.

### torch

An op-for-op tensor mirror of the scratch — same bisection for each bandwidth, same 1e-12 floors, same early exaggeration — with the init drawn from `np.random.default_rng(seed)` so both lanes take identical steps. **What torch adds:** autograd as referee — the checks differentiate KL(P‖Q) directly and confirm the hand-derived gradient to ~1e-12. The fixture stays at learning rate 10 for 120 iterations: gentle enough that float64 rounding differences do not amplify chaotically across implementations.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Mirror every guard: the 1e-12 floors, the max subtraction, the 50 bisections.
# 2. Draw the init with np.random.default_rng(seed) and torch.as_tensor it — float64.
# 3. fill_diagonal_(0.0) is torch's np.fill_diagonal; use it on D and on inv.
# 4. The training loop uses the hand gradient; autograd only certifies it in the checks.
# 5. For the autograd check, zero inv's diagonal with (1 - eye), not in place.


class TSNEScratch:
    """t-SNE on tensors, an op-for-op mirror of the NumPy lane: same binary
    search for each sigma, same early exaggeration, same momentum update, and
    the embedding init comes from np.random.default_rng(seed) so both lanes
    take the identical first step."""

    def __init__(self, n_components=2, perplexity=30.0, n_iter=500,
                 learning_rate=200.0, momentum=0.8, seed=42):
        self.n_components = n_components
        self.perplexity = perplexity
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.seed = seed

    def _pairwise_distances(self, X):
        """Squared Euclidean distances, diagonal pinned to zero."""
        sum_sq = torch.sum(X ** 2, dim=1)
        D = sum_sq[:, None] + sum_sq[None, :] - 2.0 * X @ X.T
        D.fill_diagonal_(0.0)
        return torch.clamp(D, min=0.0)

    def _compute_perplexity_and_p(self, D_sq, target_perplexity):
        """P matrix with per-point bandwidth via the same 50-step bisection."""
        n = D_sq.shape[0]
        P = torch.zeros((n, n), dtype=torch.float64)
        target_entropy = float(np.log(target_perplexity))

        for i in range(n):
            lo, hi = 1e-10, 1e4
            p_cond = None
            for _ in range(50):
                sigma = (lo + hi) / 2.0
                dists = D_sq[i].clone()
                dists[i] = torch.inf  # exclude self
                logits = -dists / (2.0 * sigma ** 2)
                logits = logits - torch.max(logits)
                exp_logits = torch.exp(logits)
                exp_logits[i] = 0.0
                sum_exp = torch.sum(exp_logits)
                if float(sum_exp) == 0.0:
                    lo = sigma
                    continue
                p_cond = exp_logits / sum_exp

                p_safe = torch.clamp(p_cond, min=1e-12)
                entropy = -torch.sum(p_cond * torch.log(p_safe))
                if float(entropy) > target_entropy:
                    hi = sigma
                else:
                    lo = sigma

            P[i] = p_cond

        P = (P + P.T) / (2.0 * n)
        return torch.clamp(P, min=1e-12)

    def _compute_q(self, Y):
        """Student-t kernel in the embedding, normalised over all pairs."""
        D_sq = self._pairwise_distances(Y)
        inv = 1.0 / (1.0 + D_sq)
        inv.fill_diagonal_(0.0)
        Q = inv / torch.sum(inv)
        return torch.clamp(Q, min=1e-12), inv

    def _kl_divergence(self, P, Q):
        return float(torch.sum(P * torch.log(P / Q)))

    def fit_transform(self, X):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        n = Xt.shape[0]
        rng_tsne = np.random.default_rng(self.seed)

        D_sq = self._pairwise_distances(Xt)
        P = self._compute_perplexity_and_p(D_sq, self.perplexity)
        P_exag = P * 4.0

        Y = torch.as_tensor(rng_tsne.normal(scale=1e-4, size=(n, self.n_components)))
        velocity = torch.zeros_like(Y)
        self.kl_history_ = []

        for it in range(self.n_iter):
            P_use = P_exag if it < 100 else P
            Q, inv_dist = self._compute_q(Y)
            self.kl_history_.append(self._kl_divergence(P_use, Q))

            PQ_diff = P_use - Q
            grad = torch.zeros_like(Y)
            for i in range(n):
                diff = Y[i] - Y
                grad[i] = 4.0 * torch.sum(
                    PQ_diff[i, :, None] * diff * inv_dist[i, :, None], dim=0)

            velocity = self.momentum * velocity - self.learning_rate * grad
            Y = Y + velocity
            Y = Y - Y.mean(dim=0)

        self.embedding_ = Y
        return Y


In [ ]:
# exports: Y_tsne, kl_first, kl_last
_rng_ts_eq = np.random.default_rng(1212)
X_ts_eq = np.vstack([
    _rng_ts_eq.normal(loc=[0.0, 0.0, 0.0], scale=0.4, size=(10, 3)),
    _rng_ts_eq.normal(loc=[4.0, 0.0, 0.0], scale=0.4, size=(10, 3)),
    _rng_ts_eq.normal(loc=[0.0, 4.0, 0.0], scale=0.4, size=(10, 3)),
])

_ts_eq = TSNEScratch(n_components=2, perplexity=8.0, n_iter=120,
                     learning_rate=10.0, momentum=0.8, seed=7)
Y_tsne = _ts_eq.fit_transform(X_ts_eq)
kl_first = _ts_eq.kl_history_[0]
kl_last = _ts_eq.kl_history_[-1]
print(f"KL divergence: {kl_first:.4f} -> {kl_last:.4f}")


In [ ]:
# Autograd as referee: differentiate KL(P || Q(Y)) directly and compare it
# to the notebook's hand-derived 4*(p-q)*(y_i-y_j)/(1+d^2) gradient. The
# formula is the exact gradient for the plain (unexaggerated) P, whose total
# mass is 1 — so that is where the comparison is made.
_n_chk = X_ts_eq.shape[0]
_Xt_chk = torch.as_tensor(np.asarray(X_ts_eq, dtype=float))
_P_chk = _ts_eq._compute_perplexity_and_p(_ts_eq._pairwise_distances(_Xt_chk), 8.0)
_Y0 = torch.as_tensor(np.random.default_rng(99).normal(size=(_n_chk, 2))).requires_grad_(True)
_ss = (_Y0 ** 2).sum(dim=1)
_Dc = _ss[:, None] + _ss[None, :] - 2.0 * _Y0 @ _Y0.T
_invc = (1.0 / (1.0 + _Dc)) * (1.0 - torch.eye(_n_chk, dtype=torch.float64))
_Qc = torch.clamp(_invc / _invc.sum(), min=1e-12)
_kl_chk = (_P_chk * torch.log(_P_chk / _Qc)).sum()
_kl_chk.backward()
with torch.no_grad():
    _Qd, _invd = _ts_eq._compute_q(_Y0.detach())
    _PQ = _P_chk - _Qd
    _g = torch.zeros_like(_Y0)
    for _i in range(_n_chk):
        _diff = _Y0.detach()[_i] - _Y0.detach()
        _g[_i] = 4.0 * torch.sum(_PQ[_i, :, None] * _diff * _invd[_i, :, None], dim=0)
assert float(torch.max(torch.abs(_Y0.grad - _g))) < 1e-8, \
    "autograd confirms the hand-derived t-SNE gradient"

# The fixed 120-iteration budget still has to optimise something.
_kl_hist = _ts_eq.kl_history_
assert kl_last < kl_first, "gradient descent reduces the KL divergence"

_center_chk = float(torch.max(torch.abs(Y_tsne.mean(dim=0))))
assert _center_chk < 1e-9, "the embedding stays centered"

_q_total = float(_Qd.sum())
assert bool(torch.allclose(_P_chk, _P_chk.T)) and abs(_q_total - 1.0) < 1e-9, \
    "P is symmetric and Q is a distribution over pairs"
